# DataFrame Transformations

## Goal
Apply core PySpark DataFrame operations including filtering, selecting, grouping, and aggregation.

This notebook focuses on transforming raw data into meaningful insights using distributed processing in Apache Spark.

## Step 1: Load Data from Unity Catalog Volume

We reload the dataset from the Unity Catalog volume to begin transformations. This ensures that each notebook is self-contained and reproducible.

In [0]:
df = spark.read.csv(
  "/Volumes/week4_catalog/default/week4_volume/sales_data.csv",
  header = True,
  inferSchema=True
)

## Step 2: Filter Data

In this step, we filter the dataset to focus on specific subsets of interest. Filtering is a transformation in Spark, meaning it does not execute immediately but builds a logical execution plan.

We will filter transactions where the total bill is greater than 20 to analyze higher-value orders.

In [0]:
high_value_df = df.filter(df["total_bill"] > 20)
high_value_df.show()

+----------+----+------+------+---+------+----+
|total_bill| tip|   sex|smoker|day|  time|size|
+----------+----+------+------+---+------+----+
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|
|     35.26| 5.0|Female|    No|Sun|Dinner|   4|
|     21.58|3.92|  Male|    No|Sun|Dinner|   2|
|     20.65|3.35|  Male|    No|Sat|Dinner|   3|
|     20.29|2.75|Female|    No|Sat|Dinner|   2|
|     39.42|7.58|  Male|    No|Sat|Dinner|   4|
|      21.7| 4.3|  Male|    No|Sat|Dinner|   2|
|     20.69|2.45|Female|    No|Sat|Dinner|   4|
|     24.06| 3.6|  Male|    No|Sat|Dinner|   3|
|     31.27| 5.0|  Male|    No|Sat|Dinner|   3|
|      30.4| 5.6|  Male|    No|Sun|Dinner|   4|
|     22.23| 5.0|  Male|    No|Sun|Dinner|   2|
|      32.4| 6.0|  Male|    No|Sun|Dinner|   4|
|     28.55|2.05|  Male|    No|Sun|Dinne

## Step 3: Select Relevant Columns

Selecting specific columns allows us to focus on relevant data and reduce unnecessary processing. This is especially important in large datasets to improve performance.

In [0]:
selected_df = df.select("total_bill", "tip", "day")
selected_df.show()

+----------+----+---+
|total_bill| tip|day|
+----------+----+---+
|     16.99|1.01|Sun|
|     10.34|1.66|Sun|
|     21.01| 3.5|Sun|
|     23.68|3.31|Sun|
|     24.59|3.61|Sun|
|     25.29|4.71|Sun|
|      8.77| 2.0|Sun|
|     26.88|3.12|Sun|
|     15.04|1.96|Sun|
|     14.78|3.23|Sun|
|     10.27|1.71|Sun|
|     35.26| 5.0|Sun|
|     15.42|1.57|Sun|
|     18.43| 3.0|Sun|
|     14.83|3.02|Sun|
|     21.58|3.92|Sun|
|     10.33|1.67|Sun|
|     16.29|3.71|Sun|
|     16.97| 3.5|Sun|
|     20.65|3.35|Sat|
+----------+----+---+
only showing top 20 rows


## Step 4: Grouping and Aggregation

Grouping allows us to summarize data by specific categories. Aggregations such as count, average, and sum help extract meaningful insights from the dataset.

We will group the data by day and calculate:
- total number of transactions
- average bill amount

In [0]:
from pyspark.sql.functions import avg, count

grouped_df = df.groupBy("day").agg(
  count("*").alias("total_transactions"),
  avg("total_bill").alias("avg_bill")
  )
grouped_df.show()

+----+------------------+------------------+
| day|total_transactions|          avg_bill|
+----+------------------+------------------+
| Sun|                76|21.410000000000004|
|Thur|                62|17.682741935483865|
| Sat|                87|20.441379310344825|
| Fri|                19|17.151578947368417|
+----+------------------+------------------+



## Step 5: Analysis of Transformations

### High-Value Transactions (Filtering)

After filtering transactions where `total_bill > 20`, we observe that:
- High-value transactions are primarily associated with **dinner time**.
- A large proportion of these transactions occur on **weekends (Saturday and Sunday)**.
- Higher bills are often associated with **larger group sizes (3–4 people)**.
- Tips generally increase with the total bill, indicating a proportional relationship.

This suggests that higher revenue is driven by weekend dining and larger parties.

---

### Column Selection

By selecting only relevant columns (`total_bill`, `tip`, `day`), we reduced the dataset to focus on key variables for analysis.

This step demonstrates:
- Schema simplification
- Reduced data movement (important in distributed systems)
- Improved readability for downstream operations

---

### Aggregation by Day

Grouping the dataset by `day` provides insights into transaction volume and spending behavior:

- **Saturday (87 transactions)** and **Sunday (76 transactions)** have the highest activity.
- **Thursday (62 transactions)** shows moderate activity, likely due to weekday dining patterns.
- **Friday (19 transactions)** has significantly fewer records, indicating lower engagement.

Average bill analysis:
- **Sunday (~21.41)** and **Saturday (~20.44)** have the highest average spending.
- Weekdays show lower average bills (~17–18 range).

---

### Key Insights

- Revenue is concentrated on **weekends**, both in volume and average spending.
- Customer behavior differs significantly between weekdays and weekends.
- The dataset is suitable for deeper analysis such as:
  - ranking highest spending days
  - analyzing tipping behavior
  - identifying peak business periods

These insights will guide further transformations and advanced analytics (e.g., window functions).

## Step 6: Window Functions (Advanced Analytics)

Window functions allow us to perform calculations across a set of rows related to the current row, without collapsing the dataset like aggregations do.

Unlike `groupBy`, which reduces the number of rows, window functions retain all original rows while adding new computed columns.

In this step, we will:
- rank transactions within each day
- compute cumulative metrics across partitions

This enables more advanced analytical patterns commonly used in data engineering and analytics workflows.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

### Ranking Transactions by Total Bill Within Each Day

We rank transactions within each day based on the total bill amount. This helps identify the highest-value transactions for each day.

In [0]:
window_spec = Window.partitionBy("day").orderBy(df["total_bill"].desc())

ranked_df = df.withColumn("rank", rank().over(window_spec))

ranked_df.show()

+----------+----+------+------+---+------+----+----+
|total_bill| tip|   sex|smoker|day|  time|size|rank|
+----------+----+------+------+---+------+----+----+
|     40.17|4.73|  Male|   Yes|Fri|Dinner|   4|   1|
|     28.97| 3.0|  Male|   Yes|Fri|Dinner|   2|   2|
|     27.28| 4.0|  Male|   Yes|Fri|Dinner|   2|   3|
|     22.75|3.25|Female|    No|Fri|Dinner|   2|   4|
|     22.49| 3.5|  Male|    No|Fri|Dinner|   2|   5|
|     21.01| 3.0|  Male|   Yes|Fri|Dinner|   2|   6|
|     16.32| 4.3|Female|   Yes|Fri|Dinner|   2|   7|
|     16.27| 2.5|Female|   Yes|Fri| Lunch|   2|   8|
|     15.98| 3.0|Female|    No|Fri| Lunch|   3|   9|
|     15.38| 3.0|Female|   Yes|Fri|Dinner|   2|  10|
|     13.42|3.48|Female|   Yes|Fri| Lunch|   2|  11|
|     13.42|1.58|  Male|   Yes|Fri| Lunch|   2|  11|
|     12.46| 1.5|  Male|    No|Fri|Dinner|   2|  13|
|     12.16| 2.2|  Male|   Yes|Fri| Lunch|   2|  14|
|     12.03| 1.5|  Male|   Yes|Fri|Dinner|   2|  15|
|     11.35| 2.5|Female|   Yes|Fri|Dinner|   2

### Running Total of Bills Within Each Day

We compute a running total of the total bill within each day. This helps understand cumulative spending patterns over the dataset.

In [0]:
from pyspark.sql.functions import sum
window_spec_running = Window.partitionBy("day").orderBy("total_bill")

running_df = df.withColumn(
    "running_total",
    sum("total_bill").over(window_spec_running)
)
running_df.show()

+----------+----+------+------+---+------+----+------------------+
|total_bill| tip|   sex|smoker|day|  time|size|     running_total|
+----------+----+------+------+---+------+----+------------------+
|      5.75| 1.0|Female|   Yes|Fri|Dinner|   2|              5.75|
|      8.58|1.92|  Male|   Yes|Fri| Lunch|   1|             14.33|
|     10.09| 2.0|Female|   Yes|Fri| Lunch|   2|             24.42|
|     11.35| 2.5|Female|   Yes|Fri|Dinner|   2|             35.77|
|     12.03| 1.5|  Male|   Yes|Fri|Dinner|   2|47.800000000000004|
|     12.16| 2.2|  Male|   Yes|Fri| Lunch|   2| 59.96000000000001|
|     12.46| 1.5|  Male|    No|Fri|Dinner|   2| 72.42000000000002|
|     13.42|3.48|Female|   Yes|Fri| Lunch|   2| 99.26000000000002|
|     13.42|1.58|  Male|   Yes|Fri| Lunch|   2| 99.26000000000002|
|     15.38| 3.0|Female|   Yes|Fri|Dinner|   2|114.64000000000001|
|     15.98| 3.0|Female|    No|Fri| Lunch|   3|            130.62|
|     16.27| 2.5|Female|   Yes|Fri| Lunch|   2|146.89000000000

### Analyzing Tip Behavior

To better understand customer tipping patterns, we derive a new metric: tip percentage.

Tip percentage is calculated as:
tip / total_bill

This allows us to normalize tips across different bill sizes and compare behavior more meaningfully across days and transactions.

In [0]:
from pyspark.sql.functions import col

df_with_tip_pct = df.withColumn(
    "tip_pct",
    col("tip")/col("total_bill")
)

df_with_tip_pct.show()

+----------+----+------+------+---+------+----+-------------------+
|total_bill| tip|   sex|smoker|day|  time|size|            tip_pct|
+----------+----+------+------+---+------+----+-------------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|0.05944673337257211|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|0.16054158607350097|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|0.16658733936220846|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2| 0.1397804054054054|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|0.14680764538430255|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|0.18623962040332148|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|0.22805017103762829|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|0.11607142857142858|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|0.13031914893617022|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2| 0.2185385656292287|
|     10.27|1.71|  Male|    No|Sun|Dinner|   2| 0.1665043816942551|
|     35.26| 5.0|Female|    No|Sun|Dinner|   4|0

### Average Tip Percentage by Day

We group the dataset by day to analyze how tipping behavior varies across different days of the week.

This helps identify whether customers tip more generously on specific days.

In [0]:
from pyspark.sql.functions import avg

tip_by_day = df_with_tip_pct.groupBy("day").agg(
    avg("tip_pct").alias("avg_tip_pct"),
    avg("total_bill").alias("avg_bill")
)
tip_by_day.show()

+----+-------------------+------------------+
| day|        avg_tip_pct|          avg_bill|
+----+-------------------+------------------+
| Sun|0.16689728635113457|21.410000000000004|
|Thur|0.16127563396664704|17.682741935483865|
| Sat| 0.1531517163877781|20.441379310344825|
| Fri|0.16991302873347888|17.151578947368417|
+----+-------------------+------------------+



### Combined Analysis: Volume, Revenue, and Tip Behavior

We combine multiple metrics to understand overall business performance per day:
- number of transactions
- average bill size
- average tip percentage

This provides a more complete view of customer behavior and revenue patterns.

In [0]:
from pyspark.sql.functions import count 

combined_df = df_with_tip_pct.groupBy("day").agg(
    count("*").alias("transactions"),
    avg("total_bill").alias("avg_bill"),
    avg("tip_pct").alias("avg_tip_percentage")
)
combined_df.show()

+----+------------+------------------+-------------------+
| day|transactions|          avg_bill| avg_tip_percentage|
+----+------------+------------------+-------------------+
| Sun|          76|21.410000000000004|0.16689728635113457|
|Thur|          62|17.682741935483865|0.16127563396664704|
| Sat|          87|20.441379310344825| 0.1531517163877781|
| Fri|          19|17.151578947368417|0.16991302873347888|
+----+------------+------------------+-------------------+



### Ranking Customers by Tip Percentage Within Each Day

To further analyze tipping behavior, we rank transactions within each day based on the tip percentage.

Unlike ranking by absolute tip value, ranking by tip percentage allows us to identify the most generous customers relative to their total bill. This provides a more accurate view of tipping behavior.

We use a window function partitioned by `day` and ordered by `tip_pct` in descending order. This ensures that rankings are computed independently for each day.

This type of analysis is commonly used in advanced analytics to identify top performers or outliers within specific groups.

In [0]:
window_tip  = Window.partitionBy("day").orderBy(col("tip_pct").desc())

tip_rank_df = df_with_tip_pct.withColumn(
    "tip_rank",
    rank().over(window_tip)
)
tip_rank_df.show()

+----------+----+------+------+---+------+----+-------------------+--------+
|total_bill| tip|   sex|smoker|day|  time|size|            tip_pct|tip_rank|
+----------+----+------+------+---+------+----+-------------------+--------+
|     16.32| 4.3|Female|   Yes|Fri|Dinner|   2|0.26348039215686275|       1|
|     13.42|3.48|Female|   Yes|Fri| Lunch|   2| 0.2593144560357675|       2|
|      8.58|1.92|  Male|   Yes|Fri| Lunch|   1|0.22377622377622378|       3|
|     11.35| 2.5|Female|   Yes|Fri|Dinner|   2|0.22026431718061676|       4|
|     10.09| 2.0|Female|   Yes|Fri| Lunch|   2|0.19821605550049554|       5|
|     15.38| 3.0|Female|   Yes|Fri|Dinner|   2|0.19505851755526657|       6|
|     15.98| 3.0|Female|    No|Fri| Lunch|   3|0.18773466833541927|       7|
|     12.16| 2.2|  Male|   Yes|Fri| Lunch|   2|0.18092105263157895|       8|
|      5.75| 1.0|Female|   Yes|Fri|Dinner|   2|0.17391304347826086|       9|
|     22.49| 3.5|  Male|    No|Fri|Dinner|   2|0.15562472209871056|      10|

## Step 7: Advanced Analysis of Tipping Behavior

### Tip Percentage Analysis

To better understand customer behavior, we introduced a derived metric: **tip percentage (tip_pct)**. This normalizes tips relative to the total bill and enables fair comparison across transactions.

---

### Key Observations by Day

From the aggregated results:

- **Friday has the highest average tip percentage (~0.1699)** despite having the lowest number of transactions.
- **Sunday and Saturday generate the highest revenue** (higher average bills and transaction counts), but not the highest tip percentages.
- **Saturday has the lowest average tip percentage (~0.1531)** even though it has the highest number of transactions.

This suggests that:
- Higher traffic days do not necessarily lead to more generous tipping behavior.
- Customers on less busy days (e.g., Friday) may tip more proportionally.

---

### Volume vs Behavior Trade-off

- **Saturday (87 transactions)** and **Sunday (76 transactions)** dominate in terms of volume and revenue.
- However, **Friday customers tip more generously on average**, indicating a behavioral difference rather than volume-driven trends.

This highlights an important analytical distinction:
- **Revenue drivers ≠ customer generosity**

---

### Transaction-Level Insights

From the ranked tip percentages:

- The highest tip percentages exceed **25–32%**, which are significantly above average.
- These high tip percentages often occur on **smaller bills**, confirming that:
  - tip percentage tends to be higher for lower total bills
  - larger bills tend to have more stable (lower variance) tip percentages

---

### Behavioral Patterns

- Tipping behavior varies more with **relative bill size** than absolute value.
- Smaller transactions show greater variability in tipping percentages.
- Larger transactions tend to cluster around more consistent tipping rates.

---

### Analytical Value

This dataset demonstrates how combining:
- feature engineering (`tip_pct`)
- aggregations
- window functions (ranking)

enables deeper behavioral insights beyond simple aggregations.

These techniques are commonly used in real-world analytics pipelines for:
- customer segmentation
- revenue optimization
- behavioral modeling

## Step 7: Using Spark SQL (Hybrid Querying)

In this step, we convert our DataFrame-based transformations into SQL queries.

Spark allows seamless integration between the DataFrame API and SQL by registering a DataFrame as a temporary view.

This enables us to:
- run SQL queries on distributed data
- compare SQL vs DataFrame approaches
- reuse existing SQL knowledge in a Spark environment

In [0]:
df_with_tip_pct.createOrReplaceTempView("tips_table")

### Rewriting Aggregation Using Spark SQL

In this step, we replicate the previous DataFrame aggregation using Spark SQL.

This demonstrates how the same analytical logic can be expressed using SQL syntax. Spark internally uses the same execution engine (Catalyst optimizer) for both APIs, which allows them to be used interchangeably.

This is important for:
- migrating legacy SQL workloads to Spark
- enabling collaboration between data engineers and analysts

In [0]:
spark.sql(
    """
    SELECT day,
           COUNT(*) as transactions,
           AVG(total_bill) as avg_bill
    FROM tips_table
    GROUP BY day
    """
).show()

+----+------------+------------------+
| day|transactions|          avg_bill|
+----+------------+------------------+
| Sun|          76|21.410000000000004|
|Thur|          62|17.682741935483865|
| Sat|          87|20.441379310344825|
| Fri|          19|17.151578947368417|
+----+------------+------------------+



### Comparison: DataFrame API vs Spark SQL

The results obtained using Spark SQL are identical to those produced by the DataFrame API.

This is because both approaches are translated into the same underlying execution plan by Spark's Catalyst optimizer.

This demonstrates that:
- Spark SQL and the DataFrame API are interchangeable
- Choice between them depends on developer preference and use case
- SQL is often preferred by analysts, while DataFrame API is preferred by engineers

Understanding this equivalence is important when working in mixed teams or migrating legacy SQL workflows to Spark.

### Feature Engineering and Aggregation in Spark SQL

In this step, we recreate the tip percentage calculation using Spark SQL and perform aggregation on the derived metric.

The calculation of `tip_pct` is performed at the row level in the SELECT clause, and then aggregated using AVG.

This demonstrates how SQL can be used for both feature engineering and analytical computations.

In [0]:
spark.sql(
    """
    SELECT day,
           AVG(tip / total_bill) AS avg_tip_pct,
           AVG(total_bill) AS avg_bill
    FROM tips_table
    GROUP BY day 
    """
).show()

+----+-------------------+------------------+
| day|        avg_tip_pct|          avg_bill|
+----+-------------------+------------------+
| Sun|0.16689728635113457|21.410000000000004|
|Thur|0.16127563396664704|17.682741935483865|
| Sat| 0.1531517163877781|20.441379310344825|
| Fri|0.16991302873347888|17.151578947368417|
+----+-------------------+------------------+



### Validation: SQL vs DataFrame Results

The results obtained using Spark SQL match those from the DataFrame API.

This is because both approaches are translated into the same execution plan by Spark’s Catalyst optimizer.

This demonstrates that:
- Spark SQL and the DataFrame API are functionally equivalent
- The choice between them depends on usability and team preference
- SQL is often used by analysts, while DataFrame API is more common among engineers

This interoperability is a key strength of Apache Spark.

### Window Functions in Spark SQL

In this step, we replicate the ranking logic using Spark SQL.

Window functions are defined in the SELECT clause and allow us to perform calculations across rows without reducing the dataset.

We use the RANK() function partitioned by day and ordered by tip percentage to identify the highest tipping transactions within each day.

In [0]:
spark.sql(
    """
    SELECT total_bill,
           tip,
           day, 
           tip / total_bill AS tip_pct,
           RANK() OVER (
               PARTITION BY day ORDER BY tip/total_bill DESC
           ) AS tip_rank
    FROM tips_table
    """
).show()

+----------+----+---+-------------------+--------+
|total_bill| tip|day|            tip_pct|tip_rank|
+----------+----+---+-------------------+--------+
|     16.32| 4.3|Fri|0.26348039215686275|       1|
|     13.42|3.48|Fri| 0.2593144560357675|       2|
|      8.58|1.92|Fri|0.22377622377622378|       3|
|     11.35| 2.5|Fri|0.22026431718061676|       4|
|     10.09| 2.0|Fri|0.19821605550049554|       5|
|     15.38| 3.0|Fri|0.19505851755526657|       6|
|     15.98| 3.0|Fri|0.18773466833541927|       7|
|     12.16| 2.2|Fri|0.18092105263157895|       8|
|      5.75| 1.0|Fri|0.17391304347826086|       9|
|     22.49| 3.5|Fri|0.15562472209871056|      10|
|     16.27| 2.5|Fri|0.15365703749231716|      11|
|     27.28| 4.0|Fri|0.14662756598240467|      12|
|     22.75|3.25|Fri|0.14285714285714285|      13|
|     21.01| 3.0|Fri| 0.1427891480247501|      14|
|     12.03| 1.5|Fri|0.12468827930174564|      15|
|     12.46| 1.5|Fri| 0.1203852327447833|      16|
|     40.17|4.73|Fri| 0.1177495

### Window Function Validation (SQL vs DataFrame)

The ranking results produced using Spark SQL match those generated using the DataFrame API.

This is because both implementations:
- use the same ranking logic
- are executed by Spark's Catalyst optimizer
- produce identical execution plans

However, it is important to note that:
- When multiple rows have the same value (ties), ranking functions such as RANK() assign the same rank
- The ordering of tied rows is not guaranteed unless additional sorting criteria are specified

To ensure deterministic results, a secondary ordering column can be added to the window specification.